# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook chạy trên **Kaggle** (hoặc Colab) với GPU. Mọi lần huấn luyện đi qua `train.run(Config)`; danh sách thí nghiệm,
các lựa chọn tự động và việc ghi bảng nằm trong `experiments.py`. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` của repo trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, gói kết quả).

Chạy được bằng *Save & Run All*: các lựa chọn (backbone đi tiếp, tổ hợp công thức, phương pháp suy luận) mặc định lấy
theo số liệu val, và có biến để bạn ghi đè bằng tay. Lần chạy đã xong (có `summary.json`) sẽ được bỏ qua khi chạy lại ô.

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet và GPU trong Settings.
!pip -q install timm openpyxl

import os, sys, platform

REPO_URL = "https://github.com/Hieub26/K4-Track4-Day2-Deeplearning-Advance.git"
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
CODE_DIR = f"{REPO_DIR}/submissions/2A20260269_nguyen_minh_hieu/code"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{CODE_DIR}/train.py"), "chưa thấy code/: hãy push thư mục submissions/ lên GitHub trước"
sys.path.insert(0, REPO_DIR)      # eval.py
sys.path.insert(0, CODE_DIR)      # dataset, model, losses, train, inference, benchmark, experiments

import math, json, shutil
import numpy as np, pandas as pd, torch, timm, torchvision
import matplotlib.pyplot as plt
from torch import nn
from IPython.display import display

import dataset, model as model_lib, losses, train, inference, benchmark, experiments as E
import eval as ev
from train import Config, run

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
            "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__,
            "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "KHÔNG CÓ GPU"}
print(VERSIONS)
assert DEVICE == "cuda", "cần GPU: Kaggle > Settings > Accelerator > GPU T4"

### Tải dữ liệu
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
File zip chứa thẳng các ảnh `.jpg` (không có thư mục con) nên giải nén vào `data/images/`.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/images
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls data/images | wc -l
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"

# ---- Thiết lập chung cho mọi thí nghiệm (đổi ở đây, không đổi rải rác) ----
EPOCHS = 12                 # giống nhau cho mọi backbone và mọi ablation (trừ thí nghiệm đổi số epoch)
SEEDS = (0, 1, 2)           # seed cho mốc T00 và chung kết F01
COMMON = dict(epochs=EPOCHS, batch_size=64, num_workers=min(4, os.cpu_count() or 2),
              images_dir=IMAGES_DIR, labels_dir=LABELS_DIR,
              out_dir="runs", pred_dir="predictions", curve_dir="curves")
os.makedirs("figures", exist_ok=True)
os.makedirs("tables", exist_ok=True)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Kiểm tra chia dữ liệu bắt buộc: số ảnh mỗi tập/lớp, giao rỗng, hợp đủ 17.509 ảnh, file tồn tại.
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_info = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
json.dump(split_info, open("tables/split_check.json", "w"), indent=2)

# Đối chiếu với Table 1 của bài báo (số TRÍCH DẪN từ Olsen et al. 2019)
PAPER = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]
counts = pd.DataFrame(split_info["per_class"], index=dataset.CLASS_NAMES)
counts["tổng (đếm)"] = counts.sum(axis=1)
counts["Table 1 bài báo"] = PAPER
counts["khớp"] = counts["tổng (đếm)"] == counts["Table 1 bài báo"]
display(counts)
print(f"Tỉ lệ lớp lớn nhất / nhỏ nhất (toàn bộ): {counts['tổng (đếm)'].max() / counts['tổng (đếm)'].min():.2f}")
print(f"Negatives chiếm {counts.loc['Negatives', 'tổng (đếm)'] / counts['tổng (đếm)'].sum():.1%} tổng số ảnh")
counts.to_csv("tables/class_counts.csv")

ax = counts[["train", "val", "test"]].plot.bar(figsize=(11, 4.5), width=0.8)
ax.set(title="DeepWeeds fold 0: số ảnh theo lớp và theo tập", xlabel="lớp", ylabel="số ảnh")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig("figures/eda_class_distribution.png", dpi=150); plt.show()

In [ ]:
# Nhìn dữ liệu: 4 ảnh mỗi lớp (tập train), ảnh gốc 256x256.
from PIL import Image

fig, axes = plt.subplots(9, 4, figsize=(10, 22))
for c, name in enumerate(dataset.CLASS_NAMES):
    for j, fn in enumerate(train_df[train_df.Label == c].Filename.sample(4, random_state=0)):
        im = Image.open(f"{IMAGES_DIR}/{fn}")
        axes[c, j].imshow(im); axes[c, j].axis("off")
        axes[c, j].set_title(f"{name}\n{fn} {im.size} {im.mode}", fontsize=7)
plt.tight_layout(); plt.savefig("figures/eda_samples.png", dpi=110); plt.show()

sizes = {Image.open(f"{IMAGES_DIR}/{fn}").size for fn in train_df.Filename.sample(300, random_state=0)}
print("kích thước ảnh trong 300 ảnh mẫu:", sizes)

In [ ]:
# Kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3), làm trên ResNet-50 với công thức nền.
dev = torch.device(DEVICE)
chk = Config(backbone="resnet50", **COMMON)
train.set_seed(0)                                            # 1) cố định seed

# 2) loss ban đầu của head mới xấp xỉ -ln(1/9) = 2.197 (512 ảnh val, chưa train)
net = model_lib.build_model("resnet50").to(dev)
ld = dataset.make_loader(val_df.iloc[:512], IMAGES_DIR, dataset.build_transforms(False), 64, train=False)
_, _, _, loss0 = train.evaluate(net, ld, nn.CrossEntropyLoss(), dev)
print(f"loss ban đầu = {loss0:.3f}   (ln 9 = {math.log(9):.3f})   tag = {model_lib.weights_tag(net)}")

# 3) overfit một batch nhỏ (16 ảnh, không augmentation) tới loss gần 0
small = train_df.groupby("Label").head(2).iloc[:16]
x, y, _ = next(iter(dataset.make_loader(small, IMAGES_DIR, dataset.build_transforms(False), 16, train=False)))
x, y = x.to(dev), y.to(dev)
opt = train.build_optimizer(net, chk)
model_lib.train_mode(net)
for step in range(60):
    loss = nn.functional.cross_entropy(net(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 59:
        print(f"  overfit bước {step:2d}: loss {loss.item():.4f}")
assert loss.item() < 0.05, "không overfit được một batch: kiểm tra lại code trước khi chạy thật"

# 5) chế độ train/eval: sau train_mode thì BN ở train, sau evaluate thì ở eval
model_lib.train_mode(net); assert net.training
train.evaluate(net, ld, nn.CrossEntropyLoss(), dev); assert not net.training

# lịch LR: warmup tuyến tính 1 epoch rồi cosine
opt = train.build_optimizer(net, chk); sch = train.build_scheduler(opt, chk, steps_per_epoch=len(train_df) // 64)
lrs = []
for _ in range(chk.epochs * (len(train_df) // 64)):
    lrs.append([g["lr"] for g in opt.param_groups]); opt.step(); sch.step()
plt.figure(figsize=(7, 3)); plt.plot(lrs); plt.legend([g["name"] for g in opt.param_groups])
plt.xlabel("bước"); plt.ylabel("LR"); plt.title("Lịch LR: warmup + cosine, theo nhóm tham số"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("figures/check_lr_schedule.png", dpi=120); plt.show()
print("weight decay theo nhóm:", {g["name"]: g["weight_decay"] for g in opt.param_groups})
del net, opt; torch.cuda.empty_cache()

In [ ]:
# 4) Ảnh sau augmentation (đã giải chuẩn hoá) kèm nhãn; hàng cuối là CutMix và Mixup trên cùng batch.
mean = torch.tensor(dataset.IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(dataset.IMAGENET_STD).view(3, 1, 1)
show = lambda t: (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
sample = train_df.groupby("Label").head(1)                   # mỗi lớp một ảnh
augs = ["basic", "mildcrop", "color", "trivial"]
fig, axes = plt.subplots(len(augs) + 2, 9, figsize=(18, 2.2 * (len(augs) + 2)))
torch.manual_seed(0)
for r, aug in enumerate(augs):
    ds = dataset.DeepWeedsDataset(sample, IMAGES_DIR, dataset.build_transforms(True, 224, aug))
    for c in range(9):
        img, label, _ = ds[c]
        axes[r, c].imshow(show(img)); axes[r, c].axis("off")
        axes[r, c].set_title(f"{aug}: {dataset.CLASS_NAMES[label]}", fontsize=7)
ds = dataset.DeepWeedsDataset(sample, IMAGES_DIR, dataset.build_transforms(True, 224, "basic"))
xb = torch.stack([ds[i][0] for i in range(9)]); yb = torch.tensor([ds[i][1] for i in range(9)])
for r, mode in ((len(augs), "cutmix"), (len(augs) + 1, "mixup")):
    xm, (ya, yb2, lam) = losses.mix_batch(xb, yb, 1.0, mode)
    for c in range(9):
        axes[r, c].imshow(show(xm[c])); axes[r, c].axis("off")
        axes[r, c].set_title(f"{mode} lam={lam:.2f}\n{dataset.CLASS_NAMES[ya[c]]} + {dataset.CLASS_NAMES[yb2[c]]}",
                             fontsize=6)
plt.tight_layout(); plt.savefig("figures/check_augmentations.png", dpi=110); plt.show()

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền (GUIDE.md mục 1.4), cùng seed 0, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...
Danh sách: `experiments.BACKBONES` (ResNet-50, ResNeXt-50, ConvNeXt-T, DeiT-S, Swin-T, EfficientNet-B0, MobileNetV3-L).

In [ ]:
b_cfgs = E.backbone_configs(COMMON)
E.run_all(b_cfgs)

In [ ]:
# Độ trễ sơ bộ batch 1 (FP32, có warmup + synchronize) rồi lập bảng Backbones.
b_latency = E.backbone_latency([c.backbone for c in b_cfgs], device=DEVICE)
backbones = E.backbone_table(b_cfgs, b_latency)
backbones.to_csv("tables/backbones.csv", index=False)
display(backbones.sort_values("val_macro_f1", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, lab in ((axes[0], "latency_b1_ms", "độ trễ batch 1, p50 (ms)"), (axes[1], "gmacs", "GMAC")):
    ax.scatter(backbones[col], backbones["val_macro_f1"])
    for _, r in backbones.iterrows():
        ax.annotate(r["backbone"], (r[col], r["val_macro_f1"]), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set(xlabel=lab, ylabel="macro-F1 val", title=f"Backbone: macro-F1 val theo {lab}"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("figures/backbones_tradeoff.png", dpi=150); plt.show()

# Chọn backbone đi tiếp: mặc định là macro-F1 val cao nhất. Ghi đè bằng tay nếu bạn chọn theo đánh đổi khác,
# ví dụ BEST_BACKBONE = "resnet50", và nêu lý do bằng số liệu trong báo cáo.
BEST_BACKBONE = None
BEST_BACKBONE = BEST_BACKBONE or E.pick_backbone(backbones)
print("Backbone đi tiếp:", BEST_BACKBONE)

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Trên backbone đã chọn. `T00` là công thức nền, chạy 3 seed để ước lượng nhiễu (std). Mỗi `T01…T16` chỉ khác `T00`
**một yếu tố** (danh sách ở `experiments.TRAINING`, trục A–G), chạy seed 0. Δ được so với `T00` cùng seed 0.

In [ ]:
t00_cfgs = [E.t00_config(BEST_BACKBONE, s, COMMON) for s in SEEDS]
t_cfgs = E.training_configs(BEST_BACKBONE, COMMON)      # thêm only=["T05", "T08", ...] để chạy bớt nếu thiếu ngân sách
E.run_all(t00_cfgs + t_cfgs)

In [ ]:
NOISE = E.noise_std(t00_cfgs)
training = E.training_table(t00_cfgs, t_cfgs)
print(f"Nhiễu: std macro-F1 val của T00 qua {len(SEEDS)} seed = {NOISE:.4f}")
display(training)

# Tổ hợp (T20): mỗi trục B-G lấy giá trị tốt nhất nếu Δ vượt nhiễu; ít hơn 2 trục thì lấy 2 trục có Δ lớn nhất.
# Ghi đè bằng tay nếu muốn, ví dụ COMBO_OVER = {"aug": "mildcrop", "loss": "ls", "label_smoothing": 0.1}.
COMBO_OVER, COMBO_USED = E.combo_overrides(training, NOISE)
print("Tổ hợp T20 =", " + ".join(COMBO_USED), "->", COMBO_OVER)
combo_cfg = Config(exp_id=E.COMBO_ID, backbone=BEST_BACKBONE, desc="combo", **{**COMMON, **COMBO_OVER})
E.run_all([combo_cfg])

In [ ]:
training = E.training_table(t00_cfgs, t_cfgs, combo_cfg, " + ".join(COMBO_USED) + ": " + E.differs(COMBO_OVER))
training.to_csv("tables/training.csv", index=False)
display(training.sort_values("val_macro_f1", ascending=False))

# Cộng dồn hay triệt tiêu: so Δ của tổ hợp với tổng Δ của từng yếu tố.
t = training.set_index("exp_id")
if E.COMBO_ID in t.index:
    parts = t.loc[COMBO_USED, "delta_vs_t00"]
    print(f"Δ từng yếu tố: {parts.round(4).to_dict()} | tổng = {parts.sum():+.4f} | "
          f"Δ tổ hợp T20 = {t.loc[E.COMBO_ID, 'delta_vs_t00']:+.4f} | nhiễu (std) = {NOISE:.4f}")

# Công thức cho chung kết: exp_id (seed 0) có macro-F1 val cao nhất trong T00, T03.., T20 (trục A không tính).
FINAL_RECIPE_ID = None
FINAL_RECIPE_ID = FINAL_RECIPE_ID or E.pick_final(training)
FINAL_OVER = E.overrides_of(FINAL_RECIPE_ID, COMBO_OVER)
print("Công thức chung kết:", FINAL_RECIPE_ID, FINAL_OVER or "(công thức nền)")

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**, không huấn luyện lại, với checkpoint seed 0 của công thức vừa chọn. Mã `I00` (1 view, mốc), `I01`…
Độ chính xác đo ở FP32. Độ trễ đo bằng `benchmark.py`: warmup 10 lần, `torch.cuda.synchronize()` trước và sau mỗi lần đo,
100 lần đo, **không tính tiền xử lý**, batch 1 (p50/p95/p99) và batch 32 (thông lượng).

In [ ]:
study_cfg = next(c for c in t00_cfgs + t_cfgs + [combo_cfg] if c.exp_id == FINAL_RECIPE_ID and c.seed == 0)

# Ensemble: mô hình này + 2 backbone khác tốt nhất ở Bước 1 (logit val đã lưu, cùng thứ tự file).
others = backbones[backbones.backbone != BEST_BACKBONE].sort_values("val_macro_f1", ascending=False).head(2)
members = [(f"{r.exp_id} ({r.backbone})", r.backbone, f"runs/{r.exp_id}/seed0/val_logits.npy")
           for r in others.itertuples()]
ema_summary = E.load_summary(next((c for c in t_cfgs if c.exp_id == "T14"), Config(exp_id="__none__")))

inference_table, latency_table = E.inference_study(study_cfg, DEVICE, members=members, ema_summary=ema_summary)
inference_table.to_csv("tables/inference.csv", index=False)
latency_table.to_csv("tables/latency.csv", index=False)
pd.set_option("display.max_colwidth", 120)
display(inference_table)

In [ ]:
E.plot_tradeoff(inference_table, "figures/inference_tradeoff.png",
                f"Đánh đổi macro-F1 val và độ trễ ({study_cfg.exp_id}, {BEST_BACKBONE}, {VERSIONS['gpu']})")

# Phương pháp suy luận cho chung kết: phương pháp một-mô-hình (I00-I04) có macro-F1 val cao nhất.
# Ghi đè bằng tay nếu chênh lệch nhỏ hơn nhiễu và bạn muốn cách rẻ hơn, ví dụ FINAL_METHOD = "I01".
FINAL_METHOD = None
FINAL_METHOD = FINAL_METHOD or E.pick_method(inference_table)
method_row = inference_table.set_index("exp_id").loc[FINAL_METHOD]
print(f"Suy luận chung kết: {FINAL_METHOD} = {method_row['method']} "
      f"(macro-F1 val {method_row['val_macro_f1']:.4f}, p95 batch-1 {method_row['p95_ms']:.1f} ms)")

# Cấu hình thời gian thực (RUBRIC I5): 1 lượt forward, p95 batch 1 <= 100 ms.
realtime = E.pick_realtime(inference_table, budget_ms=100.0)
print("Thời gian thực:", "KHÔNG có cấu hình nào đạt p95 <= 100 ms" if realtime is None else
      f"{realtime['exp_id']} = {realtime['method']}, p95 = {realtime['p95_ms']:.1f} ms trên {VERSIONS['gpu']}")

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Cấu hình đã chốt trên **val**: `BEST_BACKBONE` + `FINAL_OVER` + `FINAL_METHOD`.
2. Huấn luyện `F01` với 3 seed. Mốc `T00` (công thức nền + 1 view `I00`) đã có 3 seed từ Bước 2.
3. Ghi dự đoán test **một lần cho mỗi seed** (file đã có thì không chạy lại):
   `F01_seed<k>_test.csv` (sau temperature scaling, T khớp trên val), `F01_uncal_*` (chưa TS),
   `F01rt_*` (thời gian thực: cùng trọng số, 1 view), `T00_seed<k>_test.csv` (mốc).

> Sau ô này **không** quay lại sửa cấu hình rồi chạy test lần nữa.

In [ ]:
f_cfgs = [Config(exp_id="F01", backbone=BEST_BACKBONE, desc="final", seed=s, **{**COMMON, **FINAL_OVER}) for s in SEEDS]
E.run_all(f_cfgs)                                           # chỉ huấn luyện + val, chưa đụng test

# Mốc: T00 đã huấn luyện xong; bật save_test_predictions chỉ nạp checkpoint tốt nhất và chạy test 1 view.
for s in SEEDS:
    run(E.t00_config(BEST_BACKBONE, s, COMMON, save_test_predictions=True))

# Chung kết: phương pháp suy luận đã chọn + temperature scaling (T khớp trên val của từng seed).
final_info = [E.final_predictions(c, FINAL_METHOD, DEVICE) for c in f_cfgs]
display(pd.DataFrame(final_info))

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (eval.py gốc, không sửa).
for tag in ("F01", "T00", "F01rt", "F01_uncal"):
    !python {REPO_DIR}/eval.py score --pred "predictions/{tag}_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag {tag} --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận).
p95 = "" if realtime is None else f"--latency-p95-ms {realtime['p95_ms']:.2f} --latency-method proper"
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" --uncal "predictions/F01_uncal_seed*_test.csv" \
    --final-val "predictions/F01_seed*_val.csv" --val-csv {LABELS_DIR}/val_subset0.csv {p95} \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

In [ ]:
# Bảng chung kết, chỉ số theo lớp, ma trận nhầm lẫn (cộng qua 3 seed).
method_name = final_info[0].get("method", FINAL_METHOD)
final = E.final_table("predictions", LABELS_DIR, {
    "F01": E.config_description(f_cfgs[0], method_name + " + temperature scaling"),
    "T00": E.config_description(t00_cfgs[0], "1 view (I00)"),
    "F01rt": E.config_description(f_cfgs[0], "1 view, thời gian thực" + ("" if realtime is None else
                                  f" ({realtime['method']}, p95 {realtime['p95_ms']:.1f} ms)")),
    "F01_uncal": E.config_description(f_cfgs[0], method_name + ", chưa temperature scaling"),
})
per_class = E.per_class_table("predictions", LABELS_DIR, ["T00", "F01"])
final.to_csv("tables/final.csv", index=False); per_class.to_csv("tables/per_class.csv", index=False)
display(final); display(per_class)

for tag in ("F01", "T00"):
    E.plot_confusion(E.confusion_sum("predictions", LABELS_DIR, tag), f"figures/confusion_{tag}.png",
                     f"Ma trận nhầm lẫn trên test, {tag} (cộng {len(SEEDS)} seed)")

In [ ]:
# Phân tích lỗi: ảnh test bị nhầm giữa Chinee Apple (0) và Snake Weed (7), F01 seed 0.
pred = pd.read_csv("predictions/F01_seed0_test.csv")
for a, b in ((0, 7), (7, 0)):
    wrong = pred[(pred.y_true == a) & (pred.y_pred == b)].head(8)
    print(f"{dataset.CLASS_NAMES[a]} bị đoán thành {dataset.CLASS_NAMES[b]}: "
          f"{((pred.y_true == a) & (pred.y_pred == b)).sum()} / {(pred.y_true == a).sum()} ảnh")
    if len(wrong) == 0:
        continue
    fig, axes = plt.subplots(1, 8, figsize=(20, 3))
    for ax in axes: ax.axis("off")
    for ax, r in zip(axes, wrong.itertuples()):
        ax.imshow(Image.open(f"{IMAGES_DIR}/{r.Filename}"))
        ax.set_title(f"{r.Filename}\np(thật)={getattr(r, f'p{a}'):.2f} p(đoán)={getattr(r, f'p{b}'):.2f}", fontsize=7)
    plt.suptitle(f"thật: {dataset.CLASS_NAMES[a]} -> đoán: {dataset.CLASS_NAMES[b]}")
    plt.tight_layout(); plt.savefig(f"figures/errors_{a}_as_{b}.png", dpi=110); plt.show()

# Các cặp nhầm nhiều nhất (ngoài đường chéo), F01 cộng 3 seed
cm = E.confusion_sum("predictions", LABELS_DIR, "F01").astype(float)
np.fill_diagonal(cm, 0)
top = sorted(((cm[i, j], i, j) for i in range(9) for j in range(9)), reverse=True)[:8]
display(pd.DataFrame([{"thật": dataset.CLASS_NAMES[i], "đoán": dataset.CLASS_NAMES[j], "số ảnh (3 seed)": int(n)}
                      for n, i, j in top]))

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), kiểm tra `exp_id` khớp ảnh trong `curves/`, rồi gói mọi thứ cần nộp
vào `submission_outputs.zip` để tải về và giải nén vào `submissions/2A20260269_nguyen_minh_hieu/`.

In [ ]:
summary = E.summary_table(backbones, training, inference_table, final)
sheets = {"Backbones": backbones, "Training": training, "Inference": inference_table, "Final": final,
          "PerClass": per_class, "Latency": latency_table, "Summary": summary}
E.write_xlsx("results.xlsx", sheets)
display(summary)

# Kiểm tra: mỗi lần huấn luyện (B, T, F) có ảnh biểu đồ riêng; file dự đoán đủ seed.
curves = sorted(os.listdir("curves"))
for cfg in b_cfgs + t00_cfgs + t_cfgs + [combo_cfg] + f_cfgs:
    if E.load_summary(cfg) is not None:
        assert train.curve_path(cfg).exists(), f"thiếu ảnh biểu đồ của {cfg.exp_id} seed{cfg.seed}"
for tag in ("F01", "T00"):
    assert len([f for f in os.listdir("predictions") if f.startswith(f"{tag}_seed") and f.endswith("_test.csv")]) == len(SEEDS)
print(f"{len(curves)} ảnh biểu đồ, results.xlsx có {len(sheets)} sheet")

In [ ]:
# Gói kết quả để tải về (không kèm checkpoint và dataset).
OUT = "submission_outputs"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(f"{OUT}/predictions"); os.makedirs(f"{OUT}/logs")
shutil.copy("results.xlsx", OUT)
for d in ("curves", "figures", "tables", "eval_out"):
    shutil.copytree(d, f"{OUT}/{d}")
for f in os.listdir("predictions"):                       # chỉ chung kết và mốc (README mục 4)
    if f.startswith(("F01", "T00")):
        shutil.copy(f"predictions/{f}", f"{OUT}/predictions/{f}")
for root, _, files in os.walk("runs"):                    # log của mọi lần chạy: cấu hình, history, summary
    for f in files:
        if f.endswith((".json", ".csv")):
            dst = os.path.join(OUT, "logs", os.path.relpath(root, "runs"))
            os.makedirs(dst, exist_ok=True); shutil.copy(os.path.join(root, f), dst)
json.dump({"versions": VERSIONS, "epochs": EPOCHS, "seeds": list(SEEDS), "best_backbone": BEST_BACKBONE,
           "noise_std_val_macro_f1": NOISE, "combo_used": COMBO_USED, "combo_overrides": COMBO_OVER,
           "final_recipe_id": FINAL_RECIPE_ID, "final_overrides": FINAL_OVER, "final_method": FINAL_METHOD,
           "final_info": final_info, "realtime": None if realtime is None else realtime.to_dict()},
          open(f"{OUT}/choices.json", "w"), indent=2, ensure_ascii=False, default=str)
shutil.make_archive(OUT, "zip", OUT)
print("Đã tạo", OUT + ".zip", f"({os.path.getsize(OUT + '.zip') / 1e6:.1f} MB)")